Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Master Course #24512

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# Exercise: 2D Convolution Torch vs. Manual

We check what is calculated for the case of multiple input images, e.g. here for `in_channel=4`.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

## Torch Conv2D

In [ ]:
f = torch.nn.Conv2d(in_channels=4,
                    out_channels=1,
                    kernel_size=(5, 3),  # odd!
                    padding = 'same',
                    padding_mode = 'zeros',
                    bias=False)
# set up kernel weights for torch
tmp = torch.ones_like(f.weight)
tmp[0, 0, :, :] *= 1  # use simple numbers
tmp[0, 1, :, :] *= 2
tmp[0, 2, :, :] *= 3
tmp[0, 3, :, :] *= 4
# to make a difference between corr and conv, add a symmetry distractor:
tmp[0, :, 0, 0] = 0  # include a 0
tmp[0, :, -1, -1] = -1  # include a -1
f.weight = torch.nn.Parameter(tmp)

# set up kernel weights for manual convolution
kernel = np.copy(f.weight.detach().numpy()[0,:])
kernel[0, :, :] = np.fliplr(np.flipud(kernel[0, :, :]))
kernel[1, :, :] = np.fliplr(np.flipud(kernel[1, :, :]))
kernel[2, :, :] = np.fliplr(np.flipud(kernel[2, :, :]))
kernel[3, :, :] = np.fliplr(np.flipud(kernel[3, :, :]))

kernel, f.weight, kernel.shape, f.weight.shape

## Input Image

In [ ]:
X = np.arange(40)+1
X = np.reshape(X, (8,-1), order='F')
X = np.repeat(X[None, :], 4, axis=0)  
X[0, :] *= 1
X[1, :] *= -2
X[2, :] *= 3
X[3, :] *= -4
# X contains 4 images, make it a torch tensor
Xt = torch.Tensor(X[None, :])
X, Xt, X.shape, Xt.shape

In [ ]:
Yt = f(Xt)  # do torch's Conv2d operation, i.e. perform a correlation
Y = Yt[0, 0, :, :].detach().numpy()
Yt.shape, Y.shape

## Manual 2D Convolution

In [ ]:
def my_conv2D_same_zero(X, kernel):
     """ 
     2D convolution with one 2D image and one 2D filter
     padding = 'same', padding_mode = 'zeros'
     manually and probably poorly implemented
     to show what happens in essence
     note: we really calculate a convolution here
     not a correlation, because the kernel is
     horizontally and vertically flipped before
     calculation
     """

     Nrow, Ncol = kernel.shape[0], kernel.shape[1]

     Xtmp = np.zeros((2*(Nrow//2)+X.shape[0],
                      2*(Ncol//2)+X.shape[1]))
     Ytmp = np.zeros_like(Xtmp)
     Xtmp[Nrow//2:-Nrow//2+1,
          Ncol//2:-Ncol//2+1] = X
     kernel_flipped = np.flipud(np.fliplr(kernel))
     
     # manually do for all pixels
     for ri in np.arange(Nrow//2, Nrow//2+X.shape[0]): 
          for ci in np.arange(Ncol//2, Ncol//2+X.shape[1]):
               # grab small image at current pixel position ri,ci 
               tmp = Xtmp[ri-Nrow//2:ri+Nrow//2+1,
                         ci-Ncol//2:ci+Ncol//2+1]
               
               # this is the actual convolution, i.e.
               # multiply the corresponding image and kernel pixels
               # and then double sum over rows and columns
               Ytmp[ri, ci] = np.sum(kernel_flipped*tmp, axis=None)
     
     Y = Ytmp[Nrow//2:-1-Nrow//2+1,
              Ncol//2:-1-Ncol//2+1]  # cut out the actual result image
     return Y

In [ ]:
tmp = np.array([
    my_conv2D_same_zero(X[0,:], kernel[0,:]),
    my_conv2D_same_zero(X[1,:], kernel[1,:]),
    my_conv2D_same_zero(X[2,:], kernel[2,:]),
    my_conv2D_same_zero(X[3,:], kernel[3,:])])
print(tmp.shape)
Ym = np.sum(tmp, axis=0)  # summing up all filtered images
np.allclose(Y, Ym)  # yields the output image

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- feel free to use the notebooks for your own purposes
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.